# Bird Migration GPS Data — Exploratory Data Analysis
**Author:** B.Jasvanth
**ISI Kolkata IDEAS Advanced Data Science Internship 2026**

**Dataset:** White Stork GPS Tracking — Eric, Nico, Sanne · 61,920 records · Aug 2013 – Apr 2014

**Upload `bird_migration_features.csv` to `/content/` before running.**


## Setup


In [ ]:
import pandas as pd
import numpy as np
import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

COLORS = {'Eric': '#065A82', 'Nico': '#02C39A', 'Sanne': '#F4A261'}
BIRDS  = ['Eric', 'Nico', 'Sanne']


## Data Loading


In [ ]:
df = pd.read_csv('bird_migration_cleaned.csv')
df['date_time'] = pd.to_datetime(df['date_time'])
df = df.sort_values(['bird_name', 'date_time']).reset_index(drop=True)

# Temporal features
df['month']      = df['date_time'].dt.month
df['month_name'] = df['date_time'].dt.strftime('%b')
df['hour']       = df['date_time'].dt.hour

# Season encoding (Aug-Nov 2013 = outbound, Dec-Jan = wintering, Feb-Apr 2014 = return)
def encode_season(row):
    m, y = row['month'], row['date_time'].year
    if y == 2013 and m in [8,9,10,11]: return 1
    if (y == 2013 and m == 12) or (y == 2014 and m == 1): return 2
    return 3
df['season'] = df.apply(encode_season, axis=1)
season_names = {1:'Autumn migration', 2:'Wintering', 3:'Spring return'}

# is_resting flag
df['is_resting'] = (df['speed_2d'] < 1.0).astype(int)

df.info()
df[['latitude','longitude','altitude','speed_2d']].describe().round(3)


## 1. Univariate Analysis


### 1.1 Histograms


In [ ]:
fig2 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        '<b>Latitude</b>', '<b>Longitude</b>', '<b>Altitude (clipped −100 to 1000m)</b>',
        '<b>Speed 2D (clipped 0–25 m/s)</b>', '<b>Direction (0–360°)</b>', '<b>Hour of Day</b>'
    ]
)

# Latitude
fig2.add_trace(go.Histogram(x=df['latitude'], nbinsx=50, name='Latitude',
    marker_color='#1C7293', opacity=0.85,
    hovertemplate='Lat: %{x:.2f}°<br>Count: %{y}<extra></extra>'), row=1, col=1)
fig2.add_vline(x=df['latitude'].mean(), line_dash='dash', line_color='#F4A261',
    annotation_text=f"Mean: {df['latitude'].mean():.2f}°",
    annotation_position='top right', row=1, col=1)

# Longitude
fig2.add_trace(go.Histogram(x=df['longitude'], nbinsx=50, name='Longitude',
    marker_color='#02C39A', opacity=0.85,
    hovertemplate='Lon: %{x:.2f}°<br>Count: %{y}<extra></extra>'), row=1, col=2)
fig2.add_vline(x=df['longitude'].mean(), line_dash='dash', line_color='#F4A261',
    annotation_text=f"Mean: {df['longitude'].mean():.2f}°",
    annotation_position='top right', row=1, col=2)

# Altitude clipped
fig2.add_trace(go.Histogram(x=df['altitude'].clip(-100,1000), nbinsx=50,
    name='Altitude', marker_color='#065A82', opacity=0.85,
    hovertemplate='Alt: %{x}m<br>Count: %{y}<extra></extra>'), row=1, col=3)
fig2.add_vline(x=df['altitude'].mean(), line_dash='dash', line_color='#F4A261',
    annotation_text=f"Mean: {df['altitude'].mean():.0f}m", row=1, col=3)
fig2.add_vline(x=df['altitude'].median(), line_dash='dot', line_color='#E24B4A',
    annotation_text=f"Median: {df['altitude'].median():.0f}m",
    annotation_position='bottom right', row=1, col=3)

# Speed clipped — highlight resting zone
resting_pct = (df['speed_2d'].dropna() < 1.0).sum() / df['speed_2d'].dropna().shape[0] * 100
fig2.add_trace(go.Histogram(x=df['speed_2d'].dropna().clip(0,25), nbinsx=50,
    name='Speed', marker_color='#F4A261', opacity=0.85,
    hovertemplate='Speed: %{x:.2f} m/s<br>Count: %{y}<extra></extra>'), row=2, col=1)
fig2.add_vline(x=df['speed_2d'].mean(), line_dash='dash', line_color='#065A82',
    annotation_text=f"Mean: {df['speed_2d'].mean():.2f} m/s", row=2, col=1)
fig2.add_vrect(x0=0, x1=1, fillcolor='#E24B4A', opacity=0.12,
    annotation_text=f'Resting {resting_pct:.1f}%',
    annotation_position='top right', row=2, col=1)

# Direction
fig2.add_trace(go.Histogram(x=df['direction'].dropna(), nbinsx=36, name='Direction',
    marker_color='#5A7A8A', opacity=0.85,
    hovertemplate='Dir: %{x:.1f}°<br>Count: %{y}<extra></extra>'), row=2, col=2)

# Hour of day
fig2.add_trace(go.Histogram(x=df['hour'], nbinsx=24, name='Hour',
    marker_color='#7B68EE', opacity=0.85,
    hovertemplate='Hour: %{x}<br>Count: %{y}<extra></extra>'), row=2, col=3)

fig2.update_layout(
    title=dict(text='<b>EDA 2 — Feature Distributions (Histograms)</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=700, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    showlegend=False, margin=dict(t=100, b=60, l=60, r=60)
)
fig2.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig2.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig2.write_html('eda_2_histograms.html')
fig2.show()
print(f'Key finding: {resting_pct:.1f}% records are resting (speed < 1 m/s)')

### 1.2 KDE Plots


In [ ]:
features_kde = ['latitude', 'longitude', 'altitude', 'speed_2d']
fig10 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        '<b>KDE — Latitude</b>', '<b>KDE — Longitude</b>',
        '<b>KDE — Altitude (clipped)</b>', '<b>KDE — Speed 2D (clipped)</b>'
    ]
)

positions = [(1,1),(1,2),(2,1),(2,2)]
clips     = [None, None, (-100,1000), (0,25)]
colors_kde = ['#1C7293','#02C39A','#065A82','#F4A261']

for i, (feat, pos, clp, col) in enumerate(zip(features_kde, positions, clips, colors_kde)):
    for bird in BIRDS:
        vals = df[df['bird_name']==bird][feat].dropna()
        if clp:
            vals = vals.clip(*clp)
        vals_sorted = np.sort(vals.values)
        bandwidth = 1.06 * vals.std() * len(vals)**(-0.2)
        x_range = np.linspace(vals_sorted[0], vals_sorted[-1], 200)
        kde_vals = np.array([
            np.mean(np.exp(-0.5*((x - vals_sorted)/bandwidth)**2) / (bandwidth * np.sqrt(2*np.pi)))
            for x in x_range
        ])
        fig10.add_trace(go.Scatter(
            x=x_range, y=kde_vals,
            mode='lines',
            name=bird,
            line=dict(color=COLORS[bird], width=2),
            showlegend=(i==0),
            hovertemplate=f'{bird}<br>{feat}: %{{x:.2f}}<br>Density: %{{y:.4f}}<extra></extra>'
        ), row=pos[0], col=pos[1])

fig10.update_layout(
    title=dict(text='<b>EDA 10 — KDE Plots per Bird</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=680, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    margin=dict(t=100, b=60, l=60, r=60)
)
fig10.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig10.update_yaxes(title_text='Density', showgrid=True, gridcolor='#EEEEEE')
fig10.write_html('eda_10_kde.html')
fig10.show()

### 1.3 Summary Statistics


In [ ]:
display(df[['latitude','longitude','altitude','speed_2d','direction']].describe().round(3))
df['bird_name'].value_counts()


## 2. Spatial Analysis


### 2.1 GPS Trajectory Plots


In [ ]:
fig4 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'<b>{b}</b>' for b in BIRDS]
)

for col_idx, bird in enumerate(BIRDS, 1):
    bd  = df[df['bird_name']==bird].sort_values('date_time').reset_index(drop=True)
    col = COLORS[bird]

    # Path line (faint)
    fig4.add_trace(go.Scatter(
        x=bd['longitude'], y=bd['latitude'],
        mode='lines', line=dict(color=col, width=0.8),
        opacity=0.3, showlegend=False, hoverinfo='skip'
    ), row=1, col=col_idx)

    # Sampled points coloured by time index
    idx  = np.linspace(0, len(bd)-1, 3000, dtype=int)
    bd_s = bd.iloc[idx]
    fig4.add_trace(go.Scatter(
        x=bd_s['longitude'], y=bd_s['latitude'],
        mode='markers',
        marker=dict(
            size=3, color=list(range(len(bd_s))),
            colorscale='Plasma', opacity=0.5,
            showscale=(col_idx==1),
            colorbar=dict(title='Time →', len=0.8, x=-0.08) if col_idx==1 else None
        ),
        name=bird, showlegend=False,
        hovertemplate=f'Bird: {bird}<br>Lon: %{{x:.4f}}°<br>Lat: %{{y:.4f}}°<extra></extra>'
    ), row=1, col=col_idx)

    # Start marker
    fig4.add_trace(go.Scatter(
        x=[bd['longitude'].iloc[0]], y=[bd['latitude'].iloc[0]],
        mode='markers+text',
        marker=dict(size=14, color='green', symbol='circle',
                    line=dict(width=2, color='white')),
        text=['Start'], textposition='top right',
        textfont=dict(size=10, color='green'),
        showlegend=False,
        hovertemplate=f'<b>{bird} Start</b><br>Lat: {bd["latitude"].iloc[0]:.4f}°<extra></extra>'
    ), row=1, col=col_idx)

    # End marker
    fig4.add_trace(go.Scatter(
        x=[bd['longitude'].iloc[-1]], y=[bd['latitude'].iloc[-1]],
        mode='markers+text',
        marker=dict(size=14, color='red', symbol='x',
                    line=dict(width=2, color='white')),
        text=['End'], textposition='bottom right',
        textfont=dict(size=10, color='red'),
        showlegend=False,
        hovertemplate=f'<b>{bird} End</b><br>Lat: {bd["latitude"].iloc[-1]:.4f}°<extra></extra>'
    ), row=1, col=col_idx)

fig4.update_layout(
    title=dict(
        text='<b>EDA 4 — Migration Trajectory per Bird</b><br>'
             '<sup>Colour = time progression · ● Start · ✕ End</sup>',
        font=dict(size=18, color='#065A82'), x=0.5),
    height=520, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    margin=dict(t=110, b=60, l=80, r=60)
)
fig4.update_xaxes(title_text='Longitude (°)', showgrid=True, gridcolor='#EEEEEE')
fig4.update_yaxes(title_text='Latitude (°N)', showgrid=True, gridcolor='#EEEEEE')
fig4.write_html('eda_4_trajectory.html')
fig4.show()

### 2.2 Migration Routes


In [ ]:
fig8 = go.Figure()

for bird in BIRDS:
    bd  = df[df['bird_name']==bird].sort_values('date_time')
    idx = np.linspace(0, len(bd)-1, 2000, dtype=int)
    bd_s = bd.iloc[idx]
    col  = COLORS[bird]

    fig8.add_trace(go.Scattergeo(
        lat=bd_s['latitude'], lon=bd_s['longitude'],
        mode='lines+markers',
        line=dict(width=1.2, color=col),
        marker=dict(size=2, color=col, opacity=0.4),
        name=bird,
        hovertemplate=f'Bird: {bird}<br>Lat: %{{lat:.4f}}°<br>Lon: %{{lon:.4f}}°<extra></extra>'
    ))
    # Start
    fig8.add_trace(go.Scattergeo(
        lat=[bd['latitude'].iloc[0]], lon=[bd['longitude'].iloc[0]],
        mode='markers+text',
        marker=dict(size=14, color=col, symbol='circle',
                    line=dict(width=2, color='white')),
        text=[f'{bird} Start'], textposition='top right',
        textfont=dict(size=10, color=col),
        name=f'{bird} Start', showlegend=False,
        hovertemplate=f'<b>{bird} Departure</b><br>Lat: {bd["latitude"].iloc[0]:.4f}°<extra></extra>'
    ))
    # End
    fig8.add_trace(go.Scattergeo(
        lat=[bd['latitude'].iloc[-1]], lon=[bd['longitude'].iloc[-1]],
        mode='markers+text',
        marker=dict(size=14, color=col, symbol='x',
                    line=dict(width=2, color='white')),
        text=[f'{bird} End'], textposition='bottom right',
        textfont=dict(size=10, color=col),
        name=f'{bird} End', showlegend=False,
        hovertemplate=f'<b>{bird} Arrival</b><br>Lat: {bd["latitude"].iloc[-1]:.4f}°<extra></extra>'
    ))

fig8.update_layout(
    title=dict(
        text='<b>EDA 8 — Geographic Migration Map</b><br>'
             '<sup>White Stork GPS · Aug 2013–Apr 2014 · Atlantic Coast Corridor</sup>',
        font=dict(size=18, color='#065A82'), x=0.5),
    geo=dict(
        showland=True, landcolor='#F5F5F0',
        showocean=True, oceancolor='#D6EAF8',
        showcoastlines=True, coastlinecolor='#AAAAAA',
        showcountries=True, countrycolor='#CCCCCC',
        showlakes=True, lakecolor='#D6EAF8',
        showrivers=True, rivercolor='#AED6F1',
        projection_type='mercator',
        lonaxis=dict(range=[-22, 10]),
        lataxis=dict(range=[8, 58]),
        bgcolor='#F8FBFD'
    ),
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    height=680, paper_bgcolor='white',
    margin=dict(t=100, b=20, l=20, r=20)
)
fig8.write_html('eda_8_geomap.html')
fig8.show()

### 2.3 Bird-wise Movement Maps


In [ ]:
fig_bw_map = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'<b>{b} Movement Map</b>' for b in BIRDS],
    specs=[[{'type':'scattergeo'},{'type':'scattergeo'},{'type':'scattergeo'}]]
)

for col_i, bird in enumerate(BIRDS, 1):
    bd = df[df['bird_name']==bird].sort_values('date_time')
    idx = np.linspace(0, len(bd)-1, 1500, dtype=int)
    bd_s = bd.iloc[idx]
    fig_bw_map.add_trace(go.Scattergeo(
        lat=bd_s['latitude'], lon=bd_s['longitude'],
        mode='lines+markers',
        line=dict(width=1, color=COLORS[bird]),
        marker=dict(size=2, color=COLORS[bird], opacity=0.5),
        name=bird,
        hovertemplate=f'{bird}<br>Lat: %{{lat:.3f}}°<br>Lon: %{{lon:.3f}}°<extra></extra>'
    ), row=1, col=col_i)

fig_bw_map.update_geos(
    showland=True, landcolor='#F5F5F0',
    showocean=True, oceancolor='#D6EAF8',
    showcoastlines=True, coastlinecolor='#AAAAAA',
    showcountries=True, countrycolor='#CCCCCC',
    projection_type='mercator',
    lonaxis=dict(range=[-22,10]),
    lataxis=dict(range=[8,58])
)
fig_bw_map.update_layout(
    title=dict(text='<b>Bird-wise Movement Maps</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=500, paper_bgcolor='white',
    margin=dict(t=80, b=20, l=10, r=10)
)
fig_bw_map.write_html('eda_birdwise_maps.html')
fig_bw_map.show()


## 3. Temporal Analysis


### 3.1 Monthly Movement


In [ ]:
fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>GPS Records per Bird</b>',
        '<b>Null Values per Column</b>',
        '<b>Monthly GPS Records</b>'
    ]
)

# Panel 1: Records per bird
counts = df['bird_name'].value_counts().reindex(BIRDS)
fig1.add_trace(go.Bar(
    x=BIRDS, y=counts.values,
    marker_color=[COLORS[b] for b in BIRDS],
    text=[f'{int(v):,}' for v in counts.values],
    textposition='outside',
    hovertemplate='Bird: %{x}<br>Records: %{y:,}<extra></extra>',
    name='Records'
), row=1, col=1)

# Panel 2: Null values
cols  = ['longitude','latitude','altitude','speed_2d','direction','date_time']
nulls = [int(df[c].isnull().sum()) for c in cols]
ncols = ['#E24B4A' if n > 0 else '#02C39A' for n in nulls]
fig1.add_trace(go.Bar(
    x=cols, y=nulls,
    marker_color=ncols,
    text=['✓ 0' if n==0 else str(n) for n in nulls],
    textposition='outside',
    hovertemplate='Column: %{x}<br>Nulls: %{y}<extra></extra>',
    name='Nulls'
), row=1, col=2)

# Panel 3: Monthly records
month_order  = [8,9,10,11,12,1,2,3,4]
month_labels = ['Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr']
monthly = df.groupby('month').size().reindex(month_order).fillna(0)
# Peak migration = Sep,Oct,Nov
mcols = ['#02C39A' if m in [9,10,11] else '#1C7293' for m in month_order]
fig1.add_trace(go.Bar(
    x=month_labels, y=monthly.values,
    marker_color=mcols,
    text=[f'{int(v):,}' for v in monthly.values],
    textposition='outside',
    hovertemplate='Month: %{x}<br>Records: %{y:,}<extra></extra>',
    name='Monthly'
), row=1, col=3)

fig1.update_layout(
    title=dict(text='<b>EDA 1 — Dataset Overview</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=460, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    showlegend=False, margin=dict(t=100, b=60, l=60, r=60)
)
fig1.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig1.write_html('eda_1_overview.html')
fig1.show()

### 3.2 Seasonal Migration


In [ ]:
season_colors = {1: '#02C39A', 2: '#F4A261', 3: '#065A82'}
season_labels = {1: 'Autumn migration', 2: 'Wintering', 3: 'Spring return'}

fig9 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>Latitude by Season</b>',
        '<b>Speed 2D by Season</b>',
        '<b>Altitude by Season</b>'
    ]
)

features9 = [
    ('latitude',  (10, 55),    1),
    ('speed_2d',  (0, 25),     2),
    ('altitude',  (-100, 1000),3),
]

for feat, clip_val, col_idx in features9:
    for s in [1, 2, 3]:
        vals = df[df['season']==s][feat].dropna().clip(*clip_val)
        fig9.add_trace(go.Box(
            y=vals,
            name=season_labels[s],
            marker_color=season_colors[s],
            boxmean=True,
            showlegend=(col_idx==1),
            hovertemplate=f'{season_labels[s]}<br>{feat}: %{{y}}<extra></extra>'
        ), row=1, col=col_idx)

fig9.update_layout(
    title=dict(text='<b>EDA 9 — Feature Distribution by Migration Season</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=500, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    boxmode='group',
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    margin=dict(t=100, b=60, l=60, r=60)
)
fig9.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig9.write_html('eda_9_seasons.html')
fig9.show()

print('=== SEASON SEPARATION (key for classifier) ===')
for s in [1,2,3]:
    sub = df[df['season']==s]
    print(f'  Season {s} ({season_labels[s]})')
    print(f'    Latitude : {sub["latitude"].mean():.2f}°N (mean)')
    print(f'    Speed    : {sub["speed_2d"].mean():.2f} m/s (mean)')
    print(f'    Resting  : {(sub["speed_2d"]<1).sum()/len(sub)*100:.1f}% of records')

### 3.3 Daily Activity (Hour of Day)


In [ ]:
fig_daily = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'<b>{b} — Hourly Activity</b>' for b in BIRDS]
)
for col_i, bird in enumerate(BIRDS, 1):
    bd = df[df['bird_name']==bird]
    fig_daily.add_trace(go.Histogram(
        x=bd['hour'], nbinsx=24,
        marker_color=COLORS[bird], opacity=0.85,
        name=bird, showlegend=False,
        hovertemplate='Hour: %{x}<br>Count: %{y}<extra></extra>'
    ), row=1, col=col_i)
fig_daily.update_layout(
    title=dict(text='<b>Daily Activity — GPS Records by Hour of Day</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=420, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    showlegend=False, margin=dict(t=100, b=60, l=60, r=60)
)
fig_daily.update_xaxes(title_text='Hour of Day', showgrid=True, gridcolor='#EEEEEE', dtick=4)
fig_daily.update_yaxes(title_text='Record Count', showgrid=True, gridcolor='#EEEEEE')
fig_daily.write_html('eda_daily_activity.html')
fig_daily.show()


### 3.4 Temporal Trajectory


In [ ]:
fig7 = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        '<b>Latitude Over Time — Full Migration Cycle</b>',
        '<b>Speed 2D Over Time (clipped 0–20 m/s)</b>'
    ],
    vertical_spacing=0.12
)

for bird in BIRDS:
    bd  = df[df['bird_name']==bird].sort_values('date_time')
    idx = np.linspace(0, len(bd)-1, 3000, dtype=int)
    bd_s = bd.iloc[idx]

    fig7.add_trace(go.Scatter(
        x=bd_s['date_time'], y=bd_s['latitude'],
        mode='lines', line=dict(color=COLORS[bird], width=1.5),
        name=bird,
        hovertemplate=f'Bird: {bird}<br>Date: %{{x|%Y-%m-%d}}<br>Lat: %{{y:.2f}}°<extra></extra>'
    ), row=1, col=1)

    fig7.add_trace(go.Scatter(
        x=bd_s['date_time'], y=bd_s['speed_2d'].clip(0,20),
        mode='lines', line=dict(color=COLORS[bird], width=1),
        name=bird, showlegend=False,
        hovertemplate=f'Bird: {bird}<br>Date: %{{x|%Y-%m-%d}}<br>Speed: %{{y:.2f}} m/s<extra></extra>'
    ), row=2, col=1)

# Geographic reference lines
fig7.add_hline(y=52, line_dash='dot', line_color='#065A82',
    annotation_text='Netherlands (~52°N)', annotation_position='right', row=1, col=1)
fig7.add_hline(y=35, line_dash='dash', line_color='gray',
    annotation_text='Mediterranean (~35°N)', annotation_position='right', row=1, col=1)
fig7.add_hline(y=20, line_dash='dash', line_color='#F4A261',
    annotation_text='Sahara (~20°N)', annotation_position='right', row=1, col=1)
fig7.add_hline(y=13, line_dash='dot', line_color='#02C39A',
    annotation_text='W. Africa (~13°N)', annotation_position='right', row=1, col=1)

# Season shading
fig7.add_vrect(x0='2013-09-01', x1='2013-11-30', fillcolor='#02C39A', opacity=0.07,
    annotation_text='Season 1: Autumn migration', annotation_position='top left', row=1, col=1)
fig7.add_vrect(x0='2013-12-01', x1='2014-01-31', fillcolor='#F4A261', opacity=0.09,
    annotation_text='Season 2: Wintering', annotation_position='top left', row=1, col=1)
fig7.add_vrect(x0='2014-02-01', x1='2014-04-30', fillcolor='#065A82', opacity=0.07,
    annotation_text='Season 3: Spring return', annotation_position='top left', row=1, col=1)

fig7.add_hline(y=1, line_dash='dash', line_color='#E24B4A',
    annotation_text='Resting threshold (1 m/s)', annotation_position='right', row=2, col=1)

fig7.update_layout(
    title=dict(text='<b>EDA 7 — Temporal Analysis</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=780, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    margin=dict(t=80, b=60, l=60, r=160)
)
fig7.update_xaxes(title_text='Date', showgrid=True, gridcolor='#EEEEEE')
fig7.update_yaxes(title_text='Latitude (°N)', showgrid=True, gridcolor='#EEEEEE', row=1, col=1)
fig7.update_yaxes(title_text='Speed 2D (m/s)', showgrid=True, gridcolor='#EEEEEE', row=2, col=1)
fig7.write_html('eda_7_temporal.html')
fig7.show()

## 4. Bird-wise Analysis


### 4.1 Total Distance, Average Speed, Migration Duration, Resting Duration


In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

bird_stats = []
for bird in BIRDS:
    bd = df[df['bird_name']==bird].sort_values('date_time').reset_index(drop=True)
    total_dist = sum(
        haversine_km(bd.loc[i,'latitude'], bd.loc[i,'longitude'],
                     bd.loc[i+1,'latitude'], bd.loc[i+1,'longitude'])
        for i in range(len(bd)-1)
    )
    avg_speed     = bd['speed_2d'].mean()
    mig_duration  = (bd['date_time'].max() - bd['date_time'].min()).days
    rest_records  = (bd['speed_2d'] < 1.0).sum()
    rest_duration = rest_records * 1
    bird_stats.append({
        'bird': bird,
        'total_dist_km': round(total_dist, 2),
        'avg_speed_ms': round(avg_speed, 4),
        'migration_days': mig_duration,
        'resting_records': rest_records
    })
    print(f'{bird}:')
    print(f'  Total distance   : {total_dist:,.2f} km')
    print(f'  Average speed    : {avg_speed:.4f} m/s')
    print(f'  Migration period : {mig_duration} days')
    print(f'  Resting records  : {rest_records:,} ({rest_records/len(bd)*100:.1f}%)')

bird_stats_df = pd.DataFrame(bird_stats)

fig11 = make_subplots(
    rows=1, cols=4,
    subplot_titles=[
        '<b>Total Distance (km)</b>', '<b>Average Speed (m/s)</b>',
        '<b>Migration Duration (days)</b>', '<b>Resting Records</b>'
    ]
)

metrics = ['total_dist_km','avg_speed_ms','migration_days','resting_records']
for col_i, metric in enumerate(metrics, 1):
    fig11.add_trace(go.Bar(
        x=bird_stats_df['bird'],
        y=bird_stats_df[metric],
        marker_color=[COLORS[b] for b in BIRDS],
        text=[str(v) for v in bird_stats_df[metric]],
        textposition='outside',
        showlegend=False,
        hovertemplate='Bird: %{x}<br>Value: %{y}<extra></extra>'
    ), row=1, col=col_i)

fig11.update_layout(
    title=dict(text='<b>EDA 11 — Bird-wise Analysis</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=460, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    showlegend=False, margin=dict(t=100, b=60, l=60, r=60)
)
fig11.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig11.write_html('eda_11_birdwise.html')
fig11.show()

### 4.2 Box Plot Comparison per Bird


In [ ]:
fig6 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>Altitude per Bird (m)</b>',
        '<b>Speed 2D per Bird (m/s)</b>',
        '<b>Latitude per Bird (°N)</b>'
    ]
)

features = [
    ('altitude',  (-100, 1000), 1),
    ('speed_2d',  (0, 25),      2),
    ('latitude',  (10, 55),     3),
]

for feat, clip_val, col_idx in features:
    for bird in BIRDS:
        vals = df[df['bird_name']==bird][feat].dropna().clip(*clip_val)
        fig6.add_trace(go.Box(
            y=vals, name=bird,
            marker_color=COLORS[bird],
            boxmean=True,
            hovertemplate=f'Bird: {bird}<br>{feat}: %{{y}}<extra></extra>',
            showlegend=(col_idx==1)
        ), row=1, col=col_idx)

fig6.update_layout(
    title=dict(text='<b>EDA 6 — Per-Bird Feature Comparison (Box Plots)</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=500, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    boxmode='group',
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    margin=dict(t=100, b=60, l=60, r=60)
)
fig6.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig6.write_html('eda_6_boxplots.html')
fig6.show()

## 5. Correlation Analysis


### 5.1 Correlation Heatmap


In [ ]:
corr = df[['latitude','longitude','altitude','speed_2d']].corr().round(3)

fig5 = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.columns.tolist(),
    colorscale='Blues', zmin=-1, zmax=1,
    text=corr.values.round(3),
    texttemplate='<b>%{text}</b>',
    textfont=dict(size=14),
    hovertemplate='%{y} vs %{x}<br>r = %{z:.3f}<extra></extra>'
))
fig5.update_layout(
    title=dict(text='<b>EDA 5 — Pearson Correlation Matrix</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=450, width=600,
    paper_bgcolor='white',
    margin=dict(t=80, b=60, l=80, r=60)
)
fig5.write_html('eda_5_correlation.html')
fig5.show()

print('=== CORRELATION FINDINGS ===')
for f1 in corr.columns:
    for f2 in corr.columns:
        if f1 < f2:
            r = corr.loc[f1,f2]
            strength = 'Very strong' if abs(r)>0.9 else 'Moderate' if abs(r)>0.3 else 'Near zero'
            print(f'  {f1} vs {f2}: r={r:.3f} ({strength})')

### 5.2 Scatter Plots


In [ ]:
fig3 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>Lat vs Lon per bird</b>',
        '<b>Speed vs Altitude</b>',
        '<b>Path coloured by speed</b>'
    ]
)

# Panel 1: Lat vs Lon per bird
for bird in BIRDS:
    bd = df[df['bird_name'] == bird]
    fig3.add_trace(go.Scatter(
        x=bd['longitude'], y=bd['latitude'],
        mode='markers',
        marker=dict(size=2, color=COLORS[bird], opacity=0.3),
        name=bird,
        hovertemplate=f'Bird: {bird}<br>Lon: %{{x:.4f}}°<br>Lat: %{{y:.4f}}°<extra></extra>'
    ), row=1, col=1)

# Panel 2: Speed vs Altitude
fig3.add_trace(go.Scatter(
    x=df['altitude'].clip(-100,1000), y=df['speed_2d'].clip(0,25),
    mode='markers',
    marker=dict(size=2, color='#1C7293', opacity=0.15),
    name='All birds', showlegend=False,
    hovertemplate='Alt: %{x}m<br>Speed: %{y:.2f} m/s<extra></extra>'
), row=1, col=2)

# Panel 3: Path coloured by speed
fig3.add_trace(go.Scatter(
    x=df['longitude'], y=df['latitude'],
    mode='markers',
    marker=dict(
        size=2, opacity=0.3,
        color=df['speed_2d'].clip(0,15),
        colorscale='Viridis', showscale=True,
        colorbar=dict(title='Speed<br>(m/s)', x=1.02, len=0.9)
    ),
    hovertemplate='Lon: %{x:.4f}°<br>Lat: %{y:.4f}°<br>Speed: %{marker.color:.2f} m/s<extra></extra>',
    name='Speed', showlegend=False
), row=1, col=3)

fig3.update_layout(
    title=dict(text='<b>EDA 3 — Scatter Plots (Feature Relationships)</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=480, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    margin=dict(t=100, b=60, l=60, r=100)
)
for col_i, (xt, yt) in enumerate([
    ('Longitude (°)', 'Latitude (°N)'),
    ('Altitude (m)', 'Speed 2D (m/s)'),
    ('Longitude (°)', 'Latitude (°N)')
], 1):
    fig3.update_xaxes(title_text=xt, showgrid=True, gridcolor='#EEEEEE', row=1, col=col_i)
    fig3.update_yaxes(title_text=yt, showgrid=True, gridcolor='#EEEEEE', row=1, col=col_i)
fig3.write_html('eda_3_scatter.html')
fig3.show()

### 5.3 Pair Plots


In [ ]:
pair_features = ['latitude','longitude','altitude','speed_2d']
n = len(pair_features)

fig12 = make_subplots(
    rows=n, cols=n,
    subplot_titles=[f'{r} vs {c}' for r in pair_features for c in pair_features]
)

for ri, feat_y in enumerate(pair_features, 1):
    for ci, feat_x in enumerate(pair_features, 1):
        for bird in BIRDS:
            bd = df[df['bird_name']==bird]
            idx_s = np.random.choice(len(bd), size=min(500,len(bd)), replace=False)
            bd_s  = bd.iloc[idx_s]
            if feat_x == feat_y:
                fig12.add_trace(go.Histogram(
                    x=bd_s[feat_x].clip(-100,1000) if feat_x=='altitude' else bd_s[feat_x],
                    nbinsx=20,
                    marker_color=COLORS[bird], opacity=0.6,
                    name=bird, showlegend=(ri==1 and ci==1)
                ), row=ri, col=ci)
            else:
                fig12.add_trace(go.Scatter(
                    x=bd_s[feat_x], y=bd_s[feat_y],
                    mode='markers',
                    marker=dict(size=2, color=COLORS[bird], opacity=0.4),
                    name=bird, showlegend=False
                ), row=ri, col=ci)

fig12.update_layout(
    title=dict(text='<b>EDA 12 — Pair Plot Matrix</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=800, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    showlegend=True,
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#CCCCCC', borderwidth=1),
    margin=dict(t=100, b=60, l=60, r=60)
)
fig12.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig12.update_yaxes(showgrid=True, gridcolor='#EEEEEE')
fig12.write_html('eda_12_pairplot.html')
fig12.show()

## 6. Missing Data Analysis


### 6.1 Missing Value Heatmap and Missing Percentage


In [ ]:
all_cols = ['date_time','bird_name','latitude','longitude','altitude','speed_2d','direction']
missing_pct = (df[all_cols].isnull().mean() * 100).round(2)
missing_pct

sample_size = min(500, len(df))
df_sample = df[all_cols].sample(n=sample_size, random_state=42)
missing_matrix = df_sample.isnull().astype(int).values

fig13 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '<b>Missing Value Heatmap (sample 500 rows)</b>',
        '<b>Missing Percentage per Column</b>'
    ],
    column_widths=[0.65, 0.35]
)

fig13.add_trace(go.Heatmap(
    z=missing_matrix.T,
    x=[f'Row {i}' for i in range(sample_size)],
    y=all_cols,
    colorscale=[[0,'#02C39A'],[1,'#E24B4A']],
    showscale=True,
    colorbar=dict(title='Missing', tickvals=[0,1], ticktext=['Present','Missing'], len=0.5),
    hovertemplate='Column: %{y}<br>Missing: %{z}<extra></extra>'
), row=1, col=1)

bar_colors = ['#E24B4A' if p > 0 else '#02C39A' for p in missing_pct]
fig13.add_trace(go.Bar(
    x=all_cols, y=missing_pct,
    marker_color=bar_colors,
    text=[f'{p:.2f}%' for p in missing_pct],
    textposition='outside',
    showlegend=False,
    hovertemplate='Column: %{x}<br>Missing: %{y:.2f}%<extra></extra>'
), row=1, col=2)

fig13.update_layout(
    title=dict(text='<b>EDA 13 — Missing Data Analysis</b>',
               font=dict(size=18, color='#065A82'), x=0.5),
    height=500, paper_bgcolor='white', plot_bgcolor='#F8FBFD',
    margin=dict(t=100, b=80, l=80, r=60)
)
fig13.update_yaxes(title_text='Missing %', showgrid=True, gridcolor='#EEEEEE', row=1, col=2)
fig13.write_html('eda_13_missing.html')
fig13.show()


## Summary — EDA Findings


In [ ]:
# EDA findings carried into later phases:
# - 61,920 records across Eric, Nico, Sanne (Aug 2013 - Apr 2014, 258 days)
# - Nulls only in speed_2d and direction (443 rows)
# - Migration corridor: Netherlands (~52N) to West Africa (~12N), lat/lon r = 0.983
# - Season split (season 1/2/3) shows strong latitude separation - useful classifier feature
resting_pct = (df['speed_2d'].dropna() < 1.0).mean() * 100
print(f'Resting records: {resting_pct:.1f}% (speed < 1 m/s)')
